# GLM analysis of registered data - NEMES 2026 workshop

This is adapted from Cedalion tutorial notebooks. For details on each step, see Cedalion docs:

- [GLM Fingertapping Example](https://doc.ibs.tu-berlin.de/cedalion/doc/dev/examples/modeling/32_glm_fingertapping_example.html)


In [ ]:
import json

import matplotlib.pyplot as p
import numpy as np
import pandas as pd
import xarray as xr

import cedalion
import cedalion.data
import cedalion.io
import cedalion.models.glm as glm
import cedalion.nirs
import cedalion.vis.blocks as vbx
import cedalion.vis.anatomy
import cedalion.sigproc.frequency
from cedalion import units

import pyvista as pv

np.set_printoptions(suppress=True)

# What counts as a short channel?
DIST_THRESHOLD = 1.5 * units.cm 

# Example data
snirf_file = "../data/example.snirf"
fwm_dir = "../data/forward_model_registered"
tsv_filename = "../data/registration/geometry.tsv"

# Load data and handle stims

In [ ]:
# Load amp data
rec = cedalion.io.snirf.read_snirf(snirf_file)[0]

# Import registered geometry
geo3d_loaded = cedalion.io.probe_geometry.load_tsv(tsv_filename)
print("geo3d_loaded labels:", geo3d_loaded.label.values)

rec.geo3d = geo3d_loaded

amp = rec.get_timeseries()

# rename trials
keep = ["1", "2", "3", "15"]
rec.stim = rec.stim[rec.stim.trial_type.isin(keep)]
rec.stim.cd.rename_events(
    {
        "1": "control",
        "2": "Tapping/Left",
        "3": "Tapping/Right",
        "15": "start_stop",
    }
)
rec.stim = rec.stim[rec.stim.trial_type != "start_stop"]
print(rec.stim)

# Preprocessing

In [ ]:
import cedalion.sigproc.quality as quality
import cedalion.xrutils as xrutils

# Here we assess channel quality by SNR
snr_thresh = 12 # the SNR (std/mean) of a channel. Set high here for demonstration purposes

# SNR thresholding using the "snr" function of the quality subpackage
snr, snr_mask = quality.snr(rec["amp"], snr_thresh)
rec["amp"], dropped = xrutils.apply_mask(rec["amp"], snr_mask, "drop", "channel")

# show some results
print(f"channels that were set to NaN according to the SNR threshold: {dropped}")

In [ ]:
from cedalion.vis.anatomy import scalp_plot

# we can plot the values per channel in a 2D montage
wl_idx = 0

f, ax = p.subplots(1, 2, figsize=(10, 4))
for i, wl in enumerate(rec["amp"].wavelength.values):
    scalp_plot(
        rec["amp"],
        rec.geo3d,
        snr.sel(wavelength=wl),
        ax[i],
        cmap="RdYlGn",
        vmin=0,
        vmax=50,
        title=f"{wl} nm",
        cb_label="SNR",
        channel_lw=2
    )
f.tight_layout()

In [ ]:
# differential pathlength factors
dpf = xr.DataArray(
    [6, 6],
    dims="wavelength",
    coords={"wavelength": rec["amp"].wavelength},
)

# calculate optical density and concentrations
rec["od"] = cedalion.nirs.cw.int2od(rec["amp"])
rec["conc"] = cedalion.nirs.cw.od2conc(rec["od"], rec.geo3d, dpf, spectrum="prahl")

# Visualize hemoglobin timeseries

In [ ]:
ts = rec["conc"]

print(ts.channel.values)

f, ax = p.subplots(4, 1, sharex=True, figsize=(12, 6))
for i, ch in enumerate(["S3D1", "S13D9", "S3D5", "S13D11"]):
    ax[i].plot(ts.time, ts.sel(channel=ch, chromo="HbO"), "r-", label="HbO")
    ax[i].plot(ts.time, ts.sel(channel=ch, chromo="HbR"), "b-", label="HbR")
    ax[i].set_title(f"Ch. {ch}")
    vbx.plot_stim_markers(ax[i], rec.stim, y=1)
    ax[i].set_ylabel(r"$\Delta$ c / uM")

ax[0].legend(ncol=6)
ax[3].set_label("time / s")
ax[3].set_xlim(0,300)
p.tight_layout()

# Building design matrix

In [ ]:
import re 

# split time series into two based on channel distance
ts_long, ts_short = cedalion.nirs.split_long_short_channels(
    rec["conc"], rec.geo3d, distance_threshold=DIST_THRESHOLD
)

# Extra filtering for D>14
detector_ids = np.array([
    int(re.search(r"D(\d+)$", ch).group(1))
    for ch in amp.channel.values
])
extra_short = amp.channel.values[detector_ids >= 15]

# Union of both definitions
all_short = np.union1d(
    ts_short.channel.values,
    extra_short
)

all_long = np.setdiff1d(
    amp.channel.values,
    all_short
)

all_short = np.setdiff1d(all_short, np.atleast_1d(dropped))
all_long = np.setdiff1d(all_long, np.atleast_1d(dropped))

ts_short = ts.sel(channel=all_short)
ts_long = ts.sel(channel=all_long)

print(f"Long channels: {ts_long.sizes['channel']}")
print(f"Short channels: {ts_short.sizes['channel']}")

print(ts_long['channel'])
print(ts_short['channel'])

# create design matrix from hrf and short channel regressors
dms = (
    glm.design_matrix.hrf_regressors(
        ts_long, rec.stim, glm.Gamma(tau=0 * units.s, sigma=3 * units.s)
    )
    & glm.design_matrix.closest_short_channel_regressor(ts_long, ts_short, rec.geo3d)
)

# normalize short channel regressor and remove units
dms.channel_wise[0] = dms.channel_wise[0].pint.dequantify()
dms.channel_wise[0] /= dms.channel_wise[0].max("time")

# Visualizing design matrix

In [ ]:
# select common regressors
dm = dms.common
display(dm)

# using xr.DataArray.plot
f, ax = p.subplots(1,1,figsize=(12,5))
dm.sel(chromo="HbO", time=dm.time < 800).T.plot()
p.title("Shared Regressors")
p.xticks(rotation=90)
p.show()

# line plots of all regressors
f, ax = p.subplots(2,1,sharex=True, figsize=(12,5))

ch = "S16D14"

reg_colors = {"HRF control" : "r", "HRF Tapping/Left" : "g", "HRF Tapping/Right" : "b"}

for i, chromo in enumerate(["HbO", "HbR"]):
    for reg in dm.regressor.values:
        normed_reg = dm.sel(chromo=chromo, regressor=reg)
        normed_reg /= normed_reg.max()
        ax[i].plot(dm.time, normed_reg, label=reg, c=reg_colors[reg])

    for cwr in dms.channel_wise:
        for reg in cwr.regressor.values:
            normed_reg = cwr.sel(chromo=chromo, regressor=reg, channel=ch)
            normed_reg /= normed_reg.max()
            ax[i].plot(cwr.time, normed_reg, label=reg)
    vbx.plot_stim_markers(ax[i], rec.stim, y=1)
    ax[i].grid()
    ax[i].set_title(chromo)
    ax[i].set_ylim(-1.5,1.5)
f.suptitle("All Regressors for Channel " + ch)
ax[0].legend(ncol=5)
ax[0].set_xlim(0,400);

# Fitting model with AR-IRLS

In [ ]:
results = glm.fit(ts_long, dms, noise_model="ar_irls", max_jobs=1)

# access the fitted model parameters
betas = results.sm.params
display(betas)
display(betas.rename("betas").to_dataframe())

In [ ]:
f, ax = p.subplots(2, 3, figsize=(12, 8))
vlims = {"HbO" : [-1,1], "HbR" : [-0.5, 0.5]}
for i_chr, chromo in enumerate(betas.chromo.values):
    vmin, vmax = vlims[chromo]
    for i_reg, reg in enumerate(
        ["HRF Tapping/Left", "HRF Tapping/Right", "HRF control"]
    ):
        cedalion.vis.anatomy.scalp_plot(
            rec["amp"],
            rec.geo3d,
            betas.sel(chromo=chromo, regressor=reg),
            ax[i_chr, i_reg],
            min_dist=1.5 * cedalion.units.cm,
            title=f"{chromo} {reg}",
            vmin=vmin,
            vmax=vmax,
            optode_labels=True,
            cmap="RdBu_r",
            cb_label=r"$\beta$"
        )
p.tight_layout()

In [ ]:
# plot t-values of fitted model parameters
f, ax = p.subplots(2, 3, figsize=(12, 8))
vlims = {"HbO" : [-20,20], "HbR" : [-20, 20]}
for i_chr, chromo in enumerate(betas.chromo.values):
    vmin, vmax = vlims[chromo]
    for i_reg, reg in enumerate(["HRF Tapping/Left", "HRF Tapping/Right", "HRF control"]):
        cedalion.vis.anatomy.scalp_plot(
            rec["amp"],
            rec.geo3d,
            results.sm.tvalues.sel(chromo=chromo, regressor=reg),
            ax[i_chr, i_reg],
            min_dist=1.5 * cedalion.units.cm,
            title=f"{chromo} {reg}",
            vmin=vmin,
            vmax=vmax,
            optode_labels=True,
            cmap="RdBu_r",
            cb_label=r"$t$"
        )
p.tight_layout()

# Visualize using forward model and Brodmann areas

In [ ]:
# Load forward model
sensitivity_file = f"{fwm_dir}/colin27_Adot.h5"
Adot = cedalion.io.load_Adot(sensitivity_file)
print("Adot dims:", dict(Adot.sizes))

# Restrict Adot to long channels
shared_channels = np.intersect1d(Adot.channel.values, ts_long.channel.values)
Adot_long = Adot.sel(channel=shared_channels)
print(f"Long channels in Adot: {len(shared_channels)}")

In [ ]:
# Download head model along with labels
colin_ijk = cedalion.dot.get_standard_headmodel("colin27")
brodmann_voxel_label_niftii, brodmann_labels_json = cedalion.data.get_atlas_files("brodmann")

# dictionary to map numeric voxel labels in nifti to string labels
with brodmann_labels_json.open("r") as fin:
    brodmann_num2label = json.load(fin)
    brodmann_num2label = {i["index"] : i["name"] for i in brodmann_num2label["labels"]}

# show some entries of the dictionary
print(brodmann_num2label)

# label surface via MNI coords
colin_ijk_labeled = colin_ijk.assign_parcels_via_mni_coords(
    coordinate_label="parcel_brodmann",
    label_mapping=brodmann_num2label,
    voxel_label_niftii=brodmann_voxel_label_niftii,
    voxel_label_crs="mni152",
    mni_eps=5
)

# helper function to only get edges of an area
def get_parcel_boundary_edges(brain_surface, label_coord, labels=None):

    edges = brain_surface.mesh.edges_unique
    vertex_labels = brain_surface.vertices.coords[label_coord].values

    lab0 = vertex_labels[edges[:, 0]]
    lab1 = vertex_labels[edges[:, 1]]
    is_boundary = lab0 != lab1

    if labels is not None:
        labels = set(labels)
        touches_target = np.isin(lab0, list(labels)) | np.isin(lab1, list(labels))
        is_boundary &= touches_target

    return edges[is_boundary]

In [ ]:
# The order in Adot and our results need to be the same to project correctly.
# Verify this.
adot_channels = Adot_long.channel.values
results_channels = results.sm.tvalues["channel"].values
t_vals = results.sm.tvalues

print(adot_channels)
print(results_channels)
print(t_vals)

In [ ]:
t_condition = t_vals.sel(regressor="HRF Tapping/Right")
roi_labels = [
    "left_BA4",
    "right_BA4",
]

# Project using forward matrix
img_proj = (Adot_long * t_condition).sum("channel")
print(img_proj)

# Plot reconstruction
clim = (-10, 10)
p0, surf, _ = cedalion.vis.anatomy.image_recon(
    img_proj,
    colin_ijk_labeled,
    view_type="hbo_brain",
    view_position="anterior",
    off_screen=False,
    show_scalar_bar=False,
    clim=clim,
    title_str="recon",
)

# Add edges
boundary_edges = get_parcel_boundary_edges(
    colin_ijk_labeled.brain,
    "parcel_brodmann",
    labels=roi_labels,
)

lines = np.hstack(
    [np.full((len(boundary_edges), 1), 2), boundary_edges]
).astype(np.int64)

boundary_poly = pv.PolyData(
    surf.points,
    lines=lines,
)

p0.add_mesh(
    boundary_poly,
    color="lime",
    line_width=3,
    render_lines_as_tubes=False,
    pickable=False,
)

p0.show()